# Yoruba–Hausa–Igbo Multilingual ASR — XLS-R 300M + CTC

**Training data:** [`Harbidel/yoruba-hausa-igbo-asr-balanced`](https://huggingface.co/datasets/Harbidel/yoruba-hausa-igbo-asr-balanced)  
**Objective:** One shared CTC speech recognition model, with a character vocabulary that preserves African-language diacritics.  
**Outputs:** Processor, fine-tuned model checkpoints, validation/test WER and CER per language, optional Hugging Face model upload.

**Run in Google Colab with a GPU.** Start with a short smoke test before enabling full training. The dataset is large (~195 hours of balanced training audio), and XLS-R 300M training will take substantial GPU time. The repository must contain *embedded audio*, not inaccessible Colab paths. This notebook checks that before training.

**Data policy:** Vocabulary is built from training transcriptions only; no test data is used for fitting or model selection. Validation and test splits remain unchanged.

## 0. Install dependencies (run once, then restart runtime if prompted)

In [1]:
%pip -q install "transformers>=4.45,<5" "datasets>=3,<5" "accelerate>=1,<2" "huggingface_hub>=0.26" "jiwer>=3" "soundfile>=0.12" "scipy>=1.11" "sentencepiece>=0.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 1. Configuration and GPU diagnostics

For the first run, keep `SMOKE_TEST=True` and `DO_TRAIN=False`. After the data, tokenizer, and model checks succeed, set `DO_TRAIN=True` to run a short smoke test. Then set `SMOKE_TEST=False` for full training.

In [14]:
import os, gc, json, re, io, unicodedata, inspect
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Audio
from huggingface_hub import login, HfApi

DATASET_REPO = "Harbidel/yoruba-hausa-igbo-asr-balanced"
BASE_MODEL = "facebook/wav2vec2-xls-r-300m"
# Save everything (vocab, processor, checkpoints, final model) to Google Drive so a Colab disconnect loses nothing.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/multilingual_xlsr_ctc")
except ImportError:
    OUTPUT_DIR = Path("/content/multilingual_xlsr_ctc")  # not running in Colab
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REPO = "Harbidel/yoruba-hausa-igbo-xls-r-300m-ctc"

SMOKE_TEST = False  # full training
DO_TRAIN = True
PUSH_MODEL = True
SEED = 42
MAX_AUDIO_SECONDS = 20.0  # skip unusually long clips for GPU stability
# Tuned for an NVIDIA L4 (24 GB, bf16). Effective batch = 8 x 2 = 16 clips.
# If you hit CUDA out-of-memory, use TRAIN_BATCH_SIZE = 4 and GRAD_ACCUM = 4 (same effective batch).
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRAD_ACCUM = 2
LEARNING_RATE = 1e-4  # 3e-5 is too low for a freshly initialised CTC head; the model stays stuck emitting blanks/garbage
NUM_EPOCHS = 3  # ~20k optimizer steps; the LR schedule decays to zero at the end of this count
VAL_SAMPLES_PER_LANG = 100  # 300 validation clips total  # validation subset used *during* training (full validation is too slow)
NUM_WORKERS = 4

from transformers import set_seed
set_seed(SEED)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
else:
    print('WARNING: Select a GPU runtime before training.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu130
CUDA: True
GPU: NVIDIA L4
VRAM (GiB): 22.03


## 2. Hugging Face authentication

If the dataset is private, add a Colab Secret named `HF_TOKEN` with read permission and enable notebook access. Model upload requires write permission. Never paste tokens into a notebook cell.

In [3]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except (ImportError, Exception):
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Hugging Face authentication available.')
else:
    print('No token found; public datasets may still load. Private repos require HF_TOKEN.')

Hugging Face authentication available.


## 3. Load and validate the dataset

In [4]:
dataset = load_dataset(DATASET_REPO, token=HF_TOKEN)
print(dataset)
required_splits = {'train', 'validation', 'test'}
assert required_splits.issubset(dataset.keys()), f'Missing splits: {required_splits - set(dataset)}'
required_columns = {'audio', 'transcription', 'language'}
assert required_columns.issubset(dataset['train'].column_names)
for split in ['train', 'validation', 'test']:
    print(split, len(dataset[split]), pd.Series(dataset[split]['language']).value_counts().to_dict())

# Do not decode audio automatically; we'll read bytes with soundfile on demand.
dataset = dataset.cast_column('audio', Audio(decode=False))
probe = dataset['train'][0]
print('Example language:', probe['language'])
print('Example transcript:', probe['transcription'][:200])
print('Embedded audio bytes:', probe['audio'].get('bytes') is not None)
print('Audio path:', probe['audio'].get('path'))
if probe['audio'].get('bytes') is None and not os.path.isfile(str(probe['audio'].get('path'))):
    raise RuntimeError('Published audio is not embedded and its path is not accessible. Re-publish audio before training.')

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/45 [00:00<?, ?it/s]

data/train-00000-of-00045.parquet:   0%|          | 0.00/498M [00:00<?, ?B/s]

data/train-00001-of-00045.parquet:   0%|          | 0.00/496M [00:00<?, ?B/s]

data/train-00002-of-00045.parquet:   0%|          | 0.00/496M [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'language', 'source', 'speaker_id', 'duration', 'sample_id'],
        num_rows: 106356
    })
    validation: Dataset({
        features: ['audio', 'transcription', 'language', 'source', 'speaker_id', 'duration', 'sample_id'],
        num_rows: 11717
    })
    test: Dataset({
        features: ['audio', 'transcription', 'language', 'source', 'speaker_id', 'duration', 'sample_id'],
        num_rows: 11569
    })
})
train 106356 {'yo': 43720, 'ha': 34573, 'ig': 28063}
validation 11717 {'ig': 7179, 'yo': 2727, 'ha': 1811}
test 11569 {'ig': 7063, 'yo': 2660, 'ha': 1846}
Example language: ig
Example transcript: anyị na ezere ịgbanwe ọnọdụ anụmanụ n oge ime afọ ime iji belata nrụgide ime ime
Embedded audio bytes: True
Audio path: ig_train_000030198.wav


## 4. Transcript normalization and shared CTC vocabulary

Normalize Unicode to NFC, lowercase, remove punctuation, and normalize whitespace. **Keep letters, combining marks, and diacritics** (e.g., Yoruba ẹ/ọ/ṣ and tone marks; Igbo dotted vowels). The tokenizer uses `|` as the word delimiter and `[PAD]` as the CTC blank token.

In [5]:
def normalize_transcript(text):
    text = unicodedata.normalize('NFC', unicodedata.normalize('NFC', str(text)).lower())
    chars = []
    for ch in text:
        category = unicodedata.category(ch)
        if ch.isspace():
            chars.append(' ')
        elif category.startswith(('L', 'M', 'N')):
            chars.append(ch)
        else:
            chars.append(' ')
    return re.sub(r'\s+', ' ', ''.join(chars)).strip()

print('Original:', dataset['train'][0]['transcription'])
print('Normalized:', normalize_transcript(dataset['train'][0]['transcription']))

# Build vocabulary from TRAIN ONLY. Dataset text columns are small enough to iterate.
characters = set()
empty_train = 0
for text in dataset['train']['transcription']:
    normalized = normalize_transcript(text)
    if not normalized:
        empty_train += 1
    characters.update(normalized)
characters.discard(' ')
characters.discard('|')
ordered = sorted(characters)
vocab = {ch: i for i, ch in enumerate(ordered)}
vocab['|'] = len(vocab)
vocab['[UNK]'] = len(vocab)
vocab['[PAD]'] = len(vocab)
with open(OUTPUT_DIR / 'vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print('Vocabulary size:', len(vocab), 'Empty train transcripts:', empty_train)
print('Vocabulary characters:', ''.join(ordered))

Original: anyị na ezere ịgbanwe ọnọdụ anụmanụ n oge ime afọ ime iji belata nrụgide ime ime
Normalized: anyị na ezere ịgbanwe ọnọdụ anụmanụ n oge ime afọ ime iji belata nrụgide ime ime
Vocabulary size: 132 Empty train transcripts: 0
Vocabulary characters: 0123456789abcdefghijklmnopqrstuvwxyz²½¾àáâãäåçèéëìíïñòóôõöùúüýāēėęĩīıĺłńōŕşšūƅƒƙƴǎǒǔǫǹȧȯɓɔɗʊʲʷʹʻ̣̩͕̀́̂̄̆̇̌ḅḍḗḡḿṁṅṇṛṣṭạẹềểỉịọụứỳỵὸ


## 5. Build the processor and initialize XLS-R 300M

In [6]:
from transformers import (
    Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor,
    Wav2Vec2ForCTC, TrainingArguments, Trainer
)

tokenizer = Wav2Vec2CTCTokenizer(
    str(OUTPUT_DIR / 'vocab.json'),
    unk_token='[UNK]', pad_token='[PAD]', word_delimiter_token='|',
    do_lower_case=False,
)
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=16000, padding_value=0.0,
    do_normalize=True, return_attention_mask=True,
)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained(str(OUTPUT_DIR / 'processor'))

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction='mean',
    ctc_zero_infinity=True,
    attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0,
    mask_time_prob=0.05, layerdrop=0.0,  # layerdrop must be 0 with gradient checkpointing
    ignore_mismatched_sizes=True,
)
model.freeze_feature_encoder()
model.config.ctc_zero_infinity = True
model.config.use_cache = False
print('CTC vocabulary size:', model.config.vocab_size)
print('Model parameters (millions):', round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CTC vocabulary size: 134
Model parameters (millions): 315.6


## 6. Decode audio on demand

The dataset can contain embedded WAV bytes or accessible file paths. We use `soundfile` and resample to 16 kHz when needed. No second on-disk audio copy is created. Long clips and blank transcripts are filtered using metadata when available.

In [7]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd
from torch.utils.data import Dataset as TorchDataset

def decode_audio(audio_record):
    payload = audio_record.get('bytes')
    path = audio_record.get('path')
    if payload is not None:
        waveform, sr = sf.read(io.BytesIO(payload), dtype='float32', always_2d=False)
    elif path and os.path.isfile(path):
        waveform, sr = sf.read(path, dtype='float32', always_2d=False)
    else:
        raise FileNotFoundError(f'Audio not embedded and path unavailable: {path}')
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if sr != 16000:
        divisor = gcd(int(sr), 16000)
        waveform = resample_poly(waveform, 16000 // divisor, int(sr) // divisor).astype('float32')
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.size == 0:
        raise ValueError('Empty audio clip')
    # Zero-mean / unit-variance normalisation. feature_extractor.pad() does NOT do this,
    # so without it training inputs differ from inference inputs (processor(...) normalises).
    return (waveform - waveform.mean()) / np.sqrt(waveform.var() + 1e-7)

class ASRDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.ds = hf_dataset
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        example = self.ds[int(idx)]
        return {
            'speech': decode_audio(example['audio']),
            'text': normalize_transcript(example['transcription']),
        }

def valid_indices(ds):
    keep = []
    has_duration = 'duration' in ds.column_names
    durations = ds['duration'] if has_duration else [None] * len(ds)
    texts = ds['transcription']
    for i, (text, duration) in enumerate(zip(texts, durations)):
        if not normalize_transcript(text):
            continue
        if has_duration and duration is not None and not (0.1 <= float(duration) <= MAX_AUDIO_SECONDS):
            continue
        keep.append(i)
    return keep

filtered = {}
for split in ['train', 'validation', 'test']:
    keep = valid_indices(dataset[split])
    filtered[split] = dataset[split].select(keep)
    print(split, 'retained', len(keep), '/', len(dataset[split]))

# A smoke test samples a small number of recordings from each language.
def stratified_subset(ds, per_language):
    indices = []
    rng = np.random.default_rng(SEED)
    languages = np.array(ds['language'])
    for lang in sorted(set(languages)):
        pool = np.flatnonzero(languages == lang)
        indices.extend(rng.choice(pool, size=min(per_language, len(pool)), replace=False).tolist())
    return ds.select(sorted(indices))

train_hf = stratified_subset(filtered['train'], 20) if SMOKE_TEST else filtered['train']
valid_hf = stratified_subset(filtered['validation'], 10 if SMOKE_TEST else VAL_SAMPLES_PER_LANG)
train_data = ASRDataset(train_hf)
valid_data = ASRDataset(valid_hf)
print('Training rows:', len(train_data), 'Validation rows:', len(valid_data))
print('Decoded example samples:', len(train_data[0]['speech']))

train retained 105763 / 106356
validation retained 11665 / 11717
test retained 11512 / 11569
Training rows: 105763 Validation rows: 300
Decoded example samples: 167680


## 7. Dynamic CTC padding collator and WER/CER metrics

In [8]:
from dataclasses import dataclass
from jiwer import wer, cer

@dataclass
class CTCDataCollator:
    processor: Wav2Vec2Processor
    def __call__(self, examples):
        audio_inputs = [{'input_values': e['speech']} for e in examples]
        batch = self.processor.feature_extractor.pad(
            audio_inputs, padding=True, return_tensors='pt'
        )
        label_ids = [self.processor.tokenizer(e['text']).input_ids for e in examples]
        label_batch = self.processor.tokenizer.pad(
            [{'input_ids': ids} for ids in label_ids],
            padding=True, return_tensors='pt'
        )
        labels = label_batch['input_ids'].masked_fill(label_batch['attention_mask'].ne(1), -100)
        batch['labels'] = labels
        return batch

collator = CTCDataCollator(processor)

def preprocess_logits_for_metrics(logits, labels):
    # Reduce logits to token ids on the fly. Otherwise the Trainer pads variable-length logits with -100,
    # which makes argmax emit a spurious token 0, and it also stores huge float tensors in memory.
    if isinstance(logits, tuple):
        logits = logits[0]
    return torch.argmax(logits, dim=-1)

def compute_metrics(pred):
    pred_ids = np.array(pred.predictions)
    pred_ids[pred_ids == -100] = processor.tokenizer.pad_token_id
    labels = pred.label_ids.copy()
    labels[labels == -100] = processor.tokenizer.pad_token_id
    hypotheses = processor.batch_decode(pred_ids)
    references = processor.batch_decode(labels, group_tokens=False)
    return {'wer': wer(references, hypotheses), 'cer': cer(references, hypotheses)}

batch = collator([train_data[0], train_data[1]])
print({k: tuple(v.shape) for k, v in batch.items()})

{'input_values': (2, 167680), 'attention_mask': (2, 167680), 'labels': (2, 80)}


## 8. Training configuration

The defaults are conservative for 16 GB GPUs (microbatch 1, gradient accumulation 16, gradient checkpointing). For an A100/L4 you may increase batch size. A full 10-epoch run is substantial; use the smoke test first. Evaluation runs on a stratified validation subset during training, then the complete held-out test split can be evaluated later.

In [9]:
from transformers import TrainingArguments, Trainer

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=3 if SMOKE_TEST else NUM_EPOCHS,
    warmup_ratio=0.05,
    weight_decay=0.005,
    fp16=use_fp16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    # Non-reentrant checkpointing works with the frozen feature encoder (reentrant gives
    # "None of the inputs have requires_grad=True" and can silently drop gradients).
    gradient_checkpointing_kwargs={'use_reentrant': False},
    # NOTE: steps are *optimizer* steps (batches / GRAD_ACCUM). The old smoke-test values (10 / 30)
    # were never reached with 60 clips, so nothing was logged or evaluated.
    logging_steps=1 if SMOKE_TEST else 100,
    save_steps=100000 if SMOKE_TEST else 1000,  # each checkpoint (~3.7 GB) is written to Drive
    eval_steps=2 if SMOKE_TEST else 1000,
    save_total_limit=2,  # keep only the 2 newest checkpoints on Drive
    dataloader_num_workers=NUM_WORKERS,
    remove_unused_columns=False,
    report_to='none',
    seed=SEED,
    # Keep the best checkpoint (lowest validation WER) plus the newest one; older ones are deleted automatically.
    # load_best_model_at_end requires save_steps == eval_steps, so it is only enabled for the full run.
    load_best_model_at_end=not SMOKE_TEST,
    metric_for_best_model='wer',
    greater_is_better=False,
    save_strategy='steps',
    max_grad_norm=1.0,
    eval_accumulation_steps=50,
)
# Newer transformers use eval_strategy; older ones use evaluation_strategy.
params = inspect.signature(TrainingArguments.__init__).parameters
training_kwargs['eval_strategy' if 'eval_strategy' in params else 'evaluation_strategy'] = 'steps'
args = TrainingArguments(**training_kwargs)

# Trainer renamed `tokenizer` -> `processing_class` in 4.46. Pass whichever exists (never pass None for a missing arg).
trainer_params = inspect.signature(Trainer.__init__).parameters
proc_key = 'processing_class' if 'processing_class' in trainer_params else 'tokenizer'
trainer = Trainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=train_data,
    eval_dataset=valid_data,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    **{proc_key: processor},
)
print('Training configured. DO_TRAIN:', DO_TRAIN, 'SMOKE_TEST:', SMOKE_TEST)
print('Optimizer steps per epoch ~', len(train_data) // (TRAIN_BATCH_SIZE * GRAD_ACCUM))


Training configured. DO_TRAIN: True SMOKE_TEST: False
Optimizer steps per epoch ~ 6610


## 9. Train and save

**Important:** Set `DO_TRAIN=True` in the configuration cell (section 1) to run this cell; this cell no longer overrides it. For full training, also set `SMOKE_TEST=False`, then rerun the dataset-selection and Trainer cells. Checkpoints go to Google Drive and training resumes automatically from the latest one: after a disconnect, just re-run the cells from the top (sections 1 to 8) and then this cell.

In [10]:
from transformers.trainer_utils import get_last_checkpoint

if DO_TRAIN:
    if not torch.cuda.is_available():
        raise RuntimeError('GPU required for practical XLS-R 300M fine-tuning.')
    ckpt_dir = OUTPUT_DIR / 'checkpoints'
    last_ckpt = get_last_checkpoint(str(ckpt_dir)) if ckpt_dir.exists() else None
    print('Resuming from:' if last_ckpt else 'No checkpoint found; starting fresh.', last_ckpt or '')
    trainer.train(resume_from_checkpoint=last_ckpt)  # None => fresh start
    trainer.save_model(str(OUTPUT_DIR / 'final_model'))
    processor.save_pretrained(str(OUTPUT_DIR / 'final_model'))
    print('Model and processor saved to:', OUTPUT_DIR / 'final_model')
else:
    print('Training disabled. Set DO_TRAIN=True in the configuration cell.')


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 133, 'bos_token_id': 132}.


No checkpoint found; starting fresh. 


Step,Training Loss,Validation Loss,Wer,Cer
1000,1.141000,0.930018,0.733801,0.236231
2000,0.555100,0.521385,0.455673,0.142457
3000,0.470200,0.476583,0.396959,0.124951
4000,0.467400,0.395169,0.332398,0.101151
5000,0.417400,0.395316,0.333333,0.107445
6000,0.388700,0.358400,0.303158,0.093873
7000,0.367200,0.334558,0.269240,0.081973
8000,0.373200,0.325191,0.267836,0.082907
9000,0.340900,0.315223,0.265497,0.082317
10000,0.325300,0.331599,0.289825,0.088070


Model and processor saved to: /content/drive/MyDrive/multilingual_xlsr_ctc/final_model


## 10. Evaluate WER/CER by language

This evaluation is only meaningful after training. By default it uses a small per-language sample; set `EVAL_FULL_TEST=True` for the full test split. Evaluating the full test split can take considerable time.

In [11]:
EVAL_FULL_TEST = False
RUN_EVALUATION = DO_TRAIN  # meaningless on an untrained model

if RUN_EVALUATION:
    results = []
    for lang in sorted(set(filtered['test']['language'])):
        lang_ds = filtered['test'].filter(lambda l: l == lang, input_columns=['language'])  # input_columns avoids loading audio bytes for every row
        if not EVAL_FULL_TEST:
            lang_ds = stratified_subset(lang_ds, 100)
        metrics = trainer.evaluate(eval_dataset=ASRDataset(lang_ds), metric_key_prefix=f'test_{lang}')
        results.append({
            'language': lang, 'samples': len(lang_ds),
            'wer': metrics[f'test_{lang}_wer'],
            'cer': metrics[f'test_{lang}_cer'],
        })
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    results_df.to_csv(OUTPUT_DIR / 'test_metrics_by_language.csv', index=False)
else:
    print('Evaluation disabled. Set RUN_EVALUATION=True after training.')

Filter:   0%|          | 0/11512 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11512 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11512 [00:00<?, ? examples/s]

language  samples      wer      cer
      ha      100 0.120175 0.024048
      ig      100 0.311083 0.109339
      yo      100 0.272404 0.094125


## 11. Try an inference example

Run after training. Greedy CTC decoding returns a transcription without requiring the language label.

In [21]:
RUN_INFERENCE = DO_TRAIN
if RUN_INFERENCE:
    sample = filtered['test'][12]
    audio = decode_audio(sample['audio'])
    inputs = processor(audio, sampling_rate=16000, return_tensors='pt')
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        logits = model(inputs.input_values.to(device)).logits
    predicted = processor.batch_decode(torch.argmax(logits, dim=-1))[0]
    print('Language:', sample['language'])
    print('Reference:', normalize_transcript(sample['transcription']))
    print('Prediction:', predicted)
else:
    print('Inference disabled until a trained checkpoint is available.')

Language: ha
Reference: wasu masu tasiri suna amfani da shafukansu don ilmantar da jama a ba kawai don shahara ba
Prediction: wasu masu tasiri suna amfani da shafuƙansu don ilmantar da jama a ba kawai don shahara ba


## 12. Optional: upload trained model to Hugging Face

Only run after training and checking model quality. This uploads model weights and processor, not the source dataset.

In [15]:
if PUSH_MODEL:
    if not DO_TRAIN and not (OUTPUT_DIR / 'final_model').exists():
        raise RuntimeError('No saved trained model found.')
    api = HfApi(token=HF_TOKEN)
    api.create_repo(MODEL_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_folder(
        repo_id=MODEL_REPO,
        repo_type='model',
        folder_path=str(OUTPUT_DIR / 'final_model'),
        commit_message='Upload multilingual XLS-R CTC ASR model',
    )
    print('Uploaded: https://huggingface.co/' + MODEL_REPO)
else:
    print('Model upload disabled (PUSH_MODEL=False).')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...l_model/model.safetensors:   0%|          |  551kB / 1.26GB            

  ...l_model/training_args.bin:   8%|7         |   461B / 5.91kB            

Uploaded: https://huggingface.co/Harbidel/yoruba-hausa-igbo-xls-r-300m-ctc


In [22]:
import pandas as pd
from huggingface_hub import HfApi

# ---- Edit these before running ----
LICENSE = "apache-2.0"   # the base model is Apache-2.0; check your dataset's license and change if needed
INTENDED_USE = (
    "Automatic speech recognition (transcription) of Yoruba, Hausa and Igbo speech "
    "sampled at 16 kHz."
)
LIMITATIONS = (
    "- Trained on a balanced Yoruba/Hausa/Igbo dataset; performance on other accents, noisy audio, "
    "code-switching or domains not in the training data may be much worse.\n"
    "- Output is lowercase text without punctuation, with diacritics kept as in the training transcripts.\n"
    "- Not suitable for high-stakes use (legal, medical, etc.) without human review."
)
# -----------------------------------

# Results table (from the evaluation cell), if available
metrics_path = OUTPUT_DIR / "test_metrics_by_language.csv"
if metrics_path.exists():
    df = pd.read_csv(metrics_path)

    # Macro average: each language counts equally (appropriate for a balanced dataset)
    macro_wer, macro_cer = df["wer"].mean(), df["cer"].mean()
    # Sample-weighted average: languages count in proportion to the number of clips evaluated
    w = df["samples"] / df["samples"].sum()
    weighted_wer, weighted_cer = (df["wer"] * w).sum(), (df["cer"] * w).sum()

    rows = ["| Language | Samples | WER | CER |", "|---|---|---|---|"]
    for _, r in df.iterrows():
        rows.append(f"| {r['language']} | {int(r['samples'])} | {r['wer']:.3f} | {r['cer']:.3f} |")
    rows.append(f"| **Average (macro)** | {int(df['samples'].sum())} | **{macro_wer:.3f}** | **{macro_cer:.3f}** |")
    rows.append(f"| **Average (sample-weighted)** | {int(df['samples'].sum())} | **{weighted_wer:.3f}** | **{weighted_cer:.3f}** |")
    results_md = "\n".join(rows)
    if not EVAL_FULL_TEST:
        results_md += "\n\n*Evaluated on a stratified subset (up to 100 clips per language) of the test split.*"
    print(f"Average WER (macro): {macro_wer:.3f} | Average CER (macro): {macro_cer:.3f}")
else:
    results_md = "Evaluation results not yet available."

card = f"""---
language:
- yo
- ha
- ig
license: {LICENSE}
library_name: transformers
base_model: {BASE_MODEL}
datasets:
- {DATASET_REPO}
tags:
- automatic-speech-recognition
- wav2vec2
- ctc
- yoruba
- hausa
- igbo
metrics:
- wer
- cer
pipeline_tag: automatic-speech-recognition
---

# Multilingual Yoruba / Hausa / Igbo ASR (XLS-R 300M, CTC)

A single multilingual speech recognition model for **Yoruba, Hausa and Igbo**, fine-tuned from
[`{BASE_MODEL}`](https://huggingface.co/{BASE_MODEL}) with a character-level CTC head.

## Intended use
{INTENDED_USE}

## Training data
[`{DATASET_REPO}`](https://huggingface.co/datasets/{DATASET_REPO}), a balanced Yoruba/Hausa/Igbo ASR dataset.
Clips longer than {MAX_AUDIO_SECONDS:.0f} seconds were excluded. Transcripts were Unicode-normalised (NFC),
lowercased, and stripped of punctuation; diacritics and language-specific letters were kept.

## Training procedure
| Setting | Value |
|---|---|
| Base model | `{BASE_MODEL}` |
| Epochs | {NUM_EPOCHS} |
| Learning rate | {LEARNING_RATE} |
| Effective batch size | {TRAIN_BATCH_SIZE * GRAD_ACCUM} ({TRAIN_BATCH_SIZE} x {GRAD_ACCUM} accumulation) |
| Precision | bf16 / fp16 mixed precision |
| Frozen | CNN feature encoder |
| Seed | {SEED} |

## Evaluation
{results_md}

## How to use
```python
import torch, librosa
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

repo = "{MODEL_REPO}"
processor = Wav2Vec2Processor.from_pretrained(repo)
model = Wav2Vec2ForCTC.from_pretrained(repo).eval()

audio, _ = librosa.load("audio.wav", sr=16000)
inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
with torch.no_grad():
    logits = model(inputs.input_values).logits
print(processor.batch_decode(torch.argmax(logits, dim=-1))[0])
```

## Limitations
{LIMITATIONS}
"""

# Save a local copy and upload
card_path = OUTPUT_DIR / "README.md"
card_path.write_text(card, encoding="utf-8")

api = HfApi(token=HF_TOKEN)
api.create_repo(MODEL_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=str(card_path),
    path_in_repo="README.md",
    repo_id=MODEL_REPO,
    repo_type="model",
    commit_message="Add model card",
)
print("Model card uploaded: https://huggingface.co/" + MODEL_REPO)

Average WER (macro): 0.235 | Average CER (macro): 0.076
Model card uploaded: https://huggingface.co/Harbidel/yoruba-hausa-igbo-xls-r-300m-ctc
